# K-Medoids Clustering of Indonesian Provinces (IPM 2025)

Notebook ini menerapkan algoritma **K-Medoids** untuk mengelompokkan provinsi-provinsi di Indonesia berdasarkan **Indeks Pembangunan Manusia (IPM) 2025** dan komponen penyusunnya.

**Alur analisis:**
1. Data Loading
2. Exploratory Data Analysis (EDA)
3. Preprocessing (Standardisasi)
4. Penentuan Jumlah Cluster Optimal (Elbow Method & Silhouette Score)
5. K-Medoids Clustering
6. Evaluasi Hasil Cluster
7. Visualisasi Hasil
8. Interpretasi & Kesimpulan


## 1. Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.decomposition import PCA

# K-Medoids (dari package scikit-learn-extra)
from sklearn_extra.cluster import KMedoids

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

pd.set_option("display.max_columns", None)


## 2. Data Loading

Dataset berisi 38 provinsi di Indonesia dengan variabel:
- **IPM**: Indeks Pembangunan Manusia
- **UHH**: Umur Harapan Hidup (tahun)
- **HLS**: Harapan Lama Sekolah (tahun)
- **RLS**: Rata-rata Lama Sekolah (tahun)
- **Pengeluaran_per_Kapita**: Pengeluaran per Kapita Disesuaikan (Ribu Rupiah/Orang/Tahun)

Sumber: Badan Pusat Statistik (BPS), rilis 5 November 2025.


In [ ]:
df = pd.read_csv("../data/raw/ipm_provinsi_2025.csv")
df.head()


In [ ]:
print("Jumlah baris (provinsi):", df.shape[0])
print("Jumlah kolom:", df.shape[1])
df.info()


In [ ]:
# Cek missing values
df.isna().sum()


In [ ]:
# Cek duplikasi provinsi
df["Provinsi"].duplicated().sum()


## 3. Exploratory Data Analysis (EDA)

### 3.1 Statistik Deskriptif

In [ ]:
df.describe()


### 3.2 Distribusi Tiap Variabel

In [ ]:
fitur = ["IPM", "UHH", "HLS", "RLS", "Pengeluaran_per_Kapita"]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(fitur):
    sns.histplot(df[col], kde=True, ax=axes[i], color="steelblue")
    axes[i].set_title(f"Distribusi {col}")

fig.delaxes(axes[-1])
plt.tight_layout()
plt.show()


### 3.3 Korelasi Antar Variabel

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(df[fitur].corr(), annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Korelasi Antar Variabel")
plt.show()


> **Catatan:** Karena IPM adalah indeks komposit dari UHH, HLS, RLS, dan Pengeluaran per Kapita, wajar jika korelasinya tinggi antar variabel ini. Hal ini perlu diperhatikan saat interpretasi hasil cluster.

### 3.4 Provinsi dengan IPM Tertinggi dan Terendah

In [ ]:
top5 = df.nlargest(5, "IPM")[["Provinsi", "IPM"]]
bottom5 = df.nsmallest(5, "IPM")[["Provinsi", "IPM"]]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=top5, x="IPM", y="Provinsi", ax=axes[0], color="seagreen")
axes[0].set_title("5 Provinsi IPM Tertinggi")

sns.barplot(data=bottom5, x="IPM", y="Provinsi", ax=axes[1], color="indianred")
axes[1].set_title("5 Provinsi IPM Terendah")

plt.tight_layout()
plt.show()


## 4. Preprocessing: Standardisasi Data

K-Medoids berbasis jarak (distance-based), sehingga variabel dengan skala besar (misalnya Pengeluaran per Kapita dalam ribuan rupiah) dapat mendominasi hasil clustering jika tidak distandardisasi. Oleh karena itu, seluruh variabel di-standardisasi menggunakan **Z-score**.


In [ ]:
X = df[fitur].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=fitur, index=df["Provinsi"])
X_scaled_df.head()


## 5. Penentuan Jumlah Cluster Optimal

Digunakan dua metode untuk menentukan jumlah cluster (k) yang optimal:
1. **Elbow Method** — berdasarkan inertia/cost
2. **Silhouette Score** — mengukur seberapa baik pemisahan antar cluster


In [ ]:
inertia = []
silhouette_scores = []
K_range = range(2, 9)

for k in K_range:
    kmedoids = KMedoids(n_clusters=k, method="pam", random_state=42)
    labels = kmedoids.fit_predict(X_scaled)
    inertia.append(kmedoids.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(list(K_range), inertia, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Jumlah Cluster (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(K_range), silhouette_scores, marker="o", color="darkorange")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Jumlah Cluster (k)")
axes[1].set_ylabel("Silhouette Score")

plt.tight_layout()
plt.show()

for k, s in zip(K_range, silhouette_scores):
    print(f"k={k}: silhouette score = {s:.4f}")


> **TODO:** Tentukan nilai *k* optimal berdasarkan grafik di atas (titik "siku" pada Elbow Method dan/atau silhouette score tertinggi). Update variabel `k_optimal` di bawah ini sesuai hasil analisis.

In [ ]:
k_optimal = 3  # TODO: sesuaikan berdasarkan hasil Elbow Method & Silhouette Score


## 6. K-Medoids Clustering

In [ ]:
kmedoids_final = KMedoids(n_clusters=k_optimal, method="pam", random_state=42)
df["Cluster"] = kmedoids_final.fit_predict(X_scaled)

# Tampilkan medoid (provinsi yang menjadi representasi tiap cluster)
medoid_indices = kmedoids_final.medoid_indices_
print("Provinsi yang menjadi medoid tiap cluster:")
print(df.iloc[medoid_indices][["Provinsi", "Cluster"] + fitur])


In [ ]:
df[["Provinsi", "IPM", "Cluster"]].sort_values(["Cluster", "IPM"], ascending=[True, False])


## 7. Evaluasi Hasil Cluster

In [ ]:
final_silhouette = silhouette_score(X_scaled, df["Cluster"])
print(f"Silhouette Score (k={k_optimal}): {final_silhouette:.4f}")


In [ ]:
# Ringkasan rata-rata tiap variabel per cluster
cluster_summary = df.groupby("Cluster")[fitur].mean().round(2)
cluster_summary["Jumlah_Provinsi"] = df.groupby("Cluster").size()
cluster_summary


## 8. Visualisasi Hasil Cluster

### 8.1 Visualisasi 2D dengan PCA
Karena data memiliki 5 dimensi, digunakan PCA untuk mereduksi menjadi 2 dimensi agar bisa divisualisasikan.


In [ ]:
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X_scaled)

df["PCA1"] = pca_result[:, 0]
df["PCA2"] = pca_result[:, 1]

print(f"Variansi yang dijelaskan oleh 2 komponen PCA: {sum(pca.explained_variance_ratio_)*100:.2f}%")

plt.figure(figsize=(10, 7))
sns.scatterplot(data=df, x="PCA1", y="PCA2", hue="Cluster", palette="Set2", s=100)

for i, row in df.iterrows():
    plt.text(row["PCA1"]+0.05, row["PCA2"]+0.05, row["Provinsi"], fontsize=7)

plt.title(f"Visualisasi K-Medoids Clustering (k={k_optimal}) - PCA 2D")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()


### 8.2 Boxplot IPM per Cluster

In [ ]:
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x="Cluster", y="IPM", palette="Set2")
sns.stripplot(data=df, x="Cluster", y="IPM", color="black", alpha=0.5)
plt.title("Distribusi IPM per Cluster")
plt.show()


### 8.3 Silhouette Plot

In [ ]:
sil_values = silhouette_samples(X_scaled, df["Cluster"])

fig, ax = plt.subplots(figsize=(8, 6))
y_lower = 10
for i in sorted(df["Cluster"].unique()):
    cluster_sil = sil_values[df["Cluster"] == i]
    cluster_sil.sort()
    size = cluster_sil.shape[0]
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5*size, str(i))
    y_lower = y_upper + 10

ax.axvline(x=final_silhouette, color="red", linestyle="--", label="Rata-rata Silhouette Score")
ax.set_title("Silhouette Plot per Cluster")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster")
ax.legend()
plt.tight_layout()
plt.show()


## 9. Interpretasi & Kesimpulan

**TODO — isi setelah melihat hasil cluster_summary dan visualisasi:**

- **Cluster 0**: (contoh) Provinsi dengan IPM tinggi, didominasi wilayah perkotaan/Jawa-Bali...
- **Cluster 1**: (contoh) Provinsi dengan IPM sedang...
- **Cluster 2**: (contoh) Provinsi dengan IPM rendah, umumnya provinsi kepulauan/wilayah Papua...

**Insight tambahan (opsional):**
- Kaitkan hasil cluster dengan data kependudukan (luas wilayah, kepadatan penduduk) sebagai konteks tambahan.
- Bandingkan dengan kategori resmi BPS: sangat tinggi (IPM ≥ 80), tinggi (70–79,9), sedang (60–69,9), rendah (< 60).

**Kesimpulan:**
(Tulis ringkasan temuan utama dari analisis clustering ini.)
